# Qwen3.5 DataMate — benchmark trên tập train (Kaggle T4 ×2)
**Input cần gắn** (Add Input):
- dataset có thư mục **train** (có `answers.jsonl` để chấm);
- output của notebook **`kaggle_prepare.ipynb`** (model NF4 + wheel). Không có thì notebook tự tải `Qwen/Qwen3.5-9B` và lượng tử hoá lúc load (chậm hơn nhiều).

Session options: **GPU T4 ×2**. Internet chỉ cần để clone repo (và khi không có output prepare).
Chạy dài (LIMIT ≥ 5) bằng **Save Version → Save & Run All (Commit)**: bản commit chạy nền tối đa 12 giờ và giữ `/kaggle/working`.
Chạy `LIMIT = 1` trước để kiểm tra tốc độ và VRAM, rồi tăng `LIMIT` (0 = toàn bộ). Chỉ cần sửa cell đầu tiên.

In [ ]:
LIMIT = 1  # số câu; 0 = toàn bộ
SEED = 42
CONTEXT_TOKENS = 32768  # input + output mỗi bước
TEMPERATURE = 0.7  # preset non-thinking của Qwen; 0 = greedy (dễ lặp)
HARNESS = "h3"  # đổi khi đổi harness để không ghi đè/so lẫn kết quả cũ
RESUME = False
REPO_URL = "https://github.com/KhoaNotFound/ZAIC2026-BackpropEverytTime.git"
DATA_DIR = ""   # để trống: tự tìm thư mục train trong /kaggle/input
MODEL_DIR = ""  # để trống: tự tìm model đã chuẩn bị (có zaic_model.json); không thấy thì dùng hub

import os
from pathlib import Path

def find_dirs(predicate, root="/kaggle/input"):
    found = []
    for d, subdirs, files in os.walk(root):
        if predicate(set(files), set(subdirs)):
            found.append(d)
            subdirs.clear()
    return sorted(found)

if not DATA_DIR:
    candidates = find_dirs(lambda f, s: {"questions.jsonl", "answers.jsonl", "rulebook.md"} <= f and "data" in s)
    assert len(candidates) == 1, f"Cần đúng một thư mục train, tìm thấy {candidates}: gán DATA_DIR"
    DATA_DIR = candidates[0]
if not MODEL_DIR:
    models = find_dirs(lambda f, s: "zaic_model.json" in f)
    MODEL_DIR = models[0] if models else "Qwen/Qwen3.5-9B"
wheel_dirs = find_dirs(lambda f, s: "packages.json" in f)
WHEELS_DIR = wheel_dirs[0] if wheel_dirs else ""
model_tag = Path(MODEL_DIR).name.replace("/", "_")
OUTPUT = f"/kaggle/working/{model_tag}_{HARNESS}_train_limit{LIMIT}_seed{SEED}_ctx{CONTEXT_TOKENS}_t{TEMPERATURE}/predictions.jsonl"
print("DATA_DIR  =", DATA_DIR)
print("MODEL_DIR =", MODEL_DIR, "" if Path(MODEL_DIR).is_dir() else "(tải từ hub: chậm, cần Internet)")
print("WHEELS    =", WHEELS_DIR or "(không có: pip install qua mạng)")
print("OUTPUT    =", OUTPUT)

In [ ]:
import json, subprocess, sys
repo = Path("/kaggle/working/zAIC")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
# Inference chạy trong subprocess nên cài gói xong không cần restart kernel.
wheels = [str(p) for p in Path(WHEELS_DIR).iterdir() if p.suffix in (".whl", ".gz", ".zip")] if WHEELS_DIR else []
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps", *wheels], check=True)
check = "import transformers, accelerate, bitsandbytes, duckdb, openpyxl, pandas; print(transformers.__version__)"
version = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True)
if version.returncode or tuple(map(int, version.stdout.split(".")[:2])) < (5, 13):
    print("Wheel thiếu hoặc không khớp image: cài qua mạng")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)
import torch
print("transformers", subprocess.run([sys.executable, "-c", check], capture_output=True, text=True).stdout.strip(),
      "| PyTorch", torch.__version__, "| GPUs:",
      [f"{torch.cuda.get_device_name(i)} {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GiB"
       for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, "Cần 2 GPU: chọn Accelerator GPU T4 x2"

In [ ]:
env = dict(os.environ)
if Path(MODEL_DIR).is_dir():
    env["HF_HUB_OFFLINE"] = "1"  # model local: không gọi Hugging Face
command = [sys.executable, "inference.py", "--data-dir", DATA_DIR, "--limit", str(LIMIT), "--seed", str(SEED),
           "--output", OUTPUT, "--model", MODEL_DIR, "--quantization", "4bit", "--device-map", "split",
           "--context-tokens", str(CONTEXT_TOKENS), "--temperature", str(TEMPERATURE), "--evaluate"]
if RESUME:
    command.append("--resume")
subprocess.run(command + ["--dry-run"], check=True, env=env)  # kiểm tra dữ liệu, không load model
subprocess.run(command, check=True, env=env)  # chạy và chấm điểm ngay sau khi xong

In [ ]:
out = Path(OUTPUT)
for name in ["summary", "evaluation"]:
    path = out.with_suffix(f".{name}.json")
    if path.exists():
        report = json.loads(path.read_text())
        report.pop("details", None)  # từng câu: xem file evaluation.json
        print(name, json.dumps(report, ensure_ascii=False, indent=1)[:4000])
print("Artifacts:", sorted(p.name for p in out.parent.glob("predictions.*")))

Đổi số câu: sửa `LIMIT` ở cell đầu, chạy lại từ đó (thư mục output tự đổi theo model/harness/LIMIT/SEED). Chạy tiếp một đợt bị ngắt: giữ nguyên cấu hình, đặt `RESUME = True`.
Nếu dừng do lỗi (ví dụ OOM), xem `predictions.traces.jsonl`; restart kernel trước khi chạy lại với output mới. Nghi prefix cache gây lỗi: thêm `"--no-prefix-cache"` vào `command`.